# Extraction of trajectory data

The notebook is dedicated to extract tracking data from the repo https://github.com/sealneaward/nba-movement-data.git
Data consists of:
- trajectory data in json format (as .7z)
- event dataframes in csv format
- a shot dataframe, giving the exact times of the shots
    - According to the repo: "In the fixing logic, the shot time is defined as the highest acceleration point before the ball reaches it's peak, within a defined window."
      

In [1]:
import os
import json
import py7zr
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm

import matplotlib.pyplot as plt
import seaborn as sns

import math

from scipy.spatial.distance import euclidean
from scipy.spatial import ConvexHull

pd.set_option('display.max_columns', None)

In [2]:
# CONFIG

# Set Basket coordinates according to dataset coordinate system
basket_x = 89.25
basket_y = 25

# Path to cloned git repo
DATA_DIR = Path(r"C:\Users\jonas\Desktop\Weiterbildung\Projekt\nba-movement-data\data")
EVENTS_DIR = DATA_DIR / "events"
SHOTS_PATH = DATA_DIR / "shots" / "shots_fixed.csv"

# LOAD SHOTS
shots_df = pd.read_csv(SHOTS_PATH)
shots_df["GAME_ID"] = shots_df["GAME_ID"].astype(str)
shots_df["GAME_EVENT_ID"] = shots_df["GAME_EVENT_ID"].astype(str)


# Helper Functions

In [3]:
def extract_7z_file(filepath):
    '''Helper for extracting 7z files'''
    
    temp_dir = tempfile.mkdtemp()
    with py7zr.SevenZipFile(filepath, mode="r") as z:
        z.extractall(path=temp_dir)

    extracted_files = os.listdir(temp_dir)

    if len(extracted_files) == 0:
        return None

    return Path(temp_dir) / extracted_files[0]

def load_tracking_json(json_path):
    '''Helper function to load json file'''
    
    with open(json_path, "r", encoding="utf-8") as f:
        game = json.load(f)
    return game

def extract_positions(moment, postfix=''):
    '''
    Extract player and ball positions, based on a moment in the json files
    Returns: dict with ball x,y,z coordinates and player x,y coordinates, team and player id
    '''
    
    entities = moment[5]
    
    # Ball
    ball = entities[0]

    output = {
        f"ball_x{postfix}": ball[2],
        f"ball_y{postfix}": ball[3],
        f"ball_z{postfix}": ball[4],
    }

    # Players
    for idx, player in enumerate(entities[1:], start=1):
        output[f"player{idx}_team_id{postfix}"] = player[0]
        output[f"player{idx}_id{postfix}"] = player[1]

        output[f"player{idx}_x{postfix}"] = player[2]
        output[f"player{idx}_y{postfix}"] = player[3]
        #output[f"player{idx}_z"] = player[4]

    return output


In [4]:
def get_shot_window(
    game,
    period,
    shot_time,
    window_seconds=1.0
):
    """
    Returns all moments in the time window
    before the shot.
    """

    collected = []  
    for event in game["events"]:
        for moment in event["moments"]:
            try:
                moment_period = moment[0]
                game_clock = moment[2]

                if moment_period != period:
                    continue

                time_diff = game_clock - shot_time

                # Frames shortly before shot
                if 0 <= time_diff <= window_seconds:
                    collected.append(moment)
            except:
                continue

    # sort chronologically starting from shot (last moment: idx 0)
    collected = sorted(
        collected,
        key=lambda x: abs(float(x[2]) - float(shot_time))
    )
    return collected


def normalize_coordinates(x, y):
    '''Normalize coordinates to half court (47 <= x <= 94)'''
    
    if x >= 47:
        return x, y

    # flip court
    new_x = 94 - x
    new_y = 50 - y

    return new_x, new_y

def normalize_moment(moment):
    '''Normalize all coordinates of a moment'''
    
    entities = moment[5]
    
    # Ball
    ball = entities[0]
    bx, by = normalize_coordinates(
        ball[2],
        ball[3]
    )

    ball[2] = bx
    ball[3] = by

    # Players
    for player in entities[1:]:
        px, py = normalize_coordinates(
            player[2],
            player[3]
        )
        player[2] = px
        player[3] = py

    return moment

In [5]:
def compute_defender_closing_speed(moments_window, shooter_id):
    '''
    Based on moments window computes the closing speed of the closest defender
    '''

    if len(moments_window) < 5:
        return np.nan

    # Final frame
    final_moment = moments_window[0]

    final_players = build_player_dict(
        final_moment
    )

    # Extract shooter
    if shooter_id not in final_players:
        return np.nan

    shooter = final_players[shooter_id]

    sx = shooter["x"]
    sy = shooter["y"]

    shooter_team = shooter["team_id"]

    # Find nearest defender at shot
    nearest_defender_id = None
    nearest_dist = np.inf

    for pid, pdata in final_players.items():

        if pdata["team_id"] == shooter_team:
            continue

        dist = euclidean(
            (sx, sy),
            (pdata["x"], pdata["y"])
        )

        if dist < nearest_dist:

            nearest_dist = dist
            nearest_defender_id = pid

    if nearest_defender_id is None:
        return np.nan

    # Repeat for oldest frame
    old_moment = moments_window[-1]
    old_players = build_player_dict(old_moment)

    if (shooter_id not in old_players or nearest_defender_id not in old_players):
        return np.nan

    # Distance BEFORE shot
    old_shooter = old_players[shooter_id]
    old_defender = old_players[nearest_defender_id]

    old_dist = euclidean((old_shooter["x"], old_shooter["y"]), (old_defender["x"], old_defender["y"]))

    # Distance AT shot
    final_defender = final_players[nearest_defender_id]
    final_dist = euclidean((sx, sy), (final_defender["x"], final_defender["y"]))

    # Time delta
    dt = abs(old_moment[2] - final_moment[2])

    if dt == 0:
        return np.nan

    # Positive = defender closing in
    closing_speed = (old_dist - final_dist) / dt

    return closing_speed

In [6]:
def compute_ball_speed(moments_window):
    '''Compute Ball speed, based on moments window'''

    if len(moments_window) < 5:
        return np.nan

    final_moment = moments_window[0]
    old_moment = moments_window[-1]

    final_ball = final_moment[5][0]
    old_ball = old_moment[5][0]

    final_coords = (
        final_ball[2],
        final_ball[3],
        final_ball[4]
    )

    old_coords = (
        old_ball[2],
        old_ball[3],
        old_ball[4]
    )

    displacement = euclidean(
        final_coords,
        old_coords
    )

    dt = abs(
        old_moment[2] -
        final_moment[2]
    )

    if dt == 0:
        return np.nan

    return displacement / dt

In [7]:
def compute_ball_xy_speed(moments_window):
    '''Compute Ball speed in x/y coordinates, based on moments window'''

    if len(moments_window) < 5:
        return np.nan

    final_ball = moments_window[0][5][0]
    old_ball = moments_window[-1][5][0]

    displacement = euclidean(
        (final_ball[2], final_ball[3]),
        (old_ball[2], old_ball[3])
    )

    dt = abs(
        moments_window[-1][2] -
        moments_window[0][2]
    )

    if dt == 0:
        return np.nan

    return displacement / dt

In [8]:
def compute_player_speeds(moment_old, moment_new):
    """
    Compute the movement speed of all players, based on 2 moments
    Returns:
        dict with fixed keys:
        player1_speed, player2_speed, ...
    """
    old_players = build_player_dict(moment_old)
    new_players = build_player_dict(moment_new)

    dt = abs(moment_new[2] - moment_old[2])

    if dt == 0:
        return {}

    output = {}

    # iterate in the SAME ORDER extract_positions
    entities_old = moment_old[5][1:]
    entities_new = moment_new[5][1:]

    for idx, (old_p, new_p) in enumerate(zip(entities_old, entities_new), start=1):

        player_id = new_p[1]

        if player_id not in old_players or player_id not in new_players:
            output[f"player{idx}_speed"] = None
            continue

        ox, oy = old_players[player_id]["x"], old_players[player_id]["y"]
        nx, ny = new_players[player_id]["x"], new_players[player_id]["y"]

        dist = np.sqrt((nx - ox)**2 + (ny - oy)**2)

        output[f"player{idx}_speed"] = dist / dt

    return output


def compute_shooter_speed(moment_old, moment_new, shooter_id):
    """
    Compute the movement speed of the shooter, based on 2 moments
    """
    
    old_players = build_player_dict(moment_old)
    new_players = build_player_dict(moment_new)

    if shooter_id not in old_players or shooter_id not in new_players:
        return None

    old = old_players[shooter_id]
    new = new_players[shooter_id]

    dx = new["x"] - old["x"]
    dy = new["y"] - old["y"]

    dist = np.sqrt(dx**2 + dy**2)

    dt = abs(moment_new[2] - moment_old[2])

    if dt == 0:
        return None

    return dist / dt

In [9]:
def build_player_dict(moment):
    '''Create a dictionary of player positions and metadata from one tracking moment.'''
    players = {}
    for idx, p in enumerate(moment[5][1:]):

        players[p[1]] = {
            "team_id": p[0],
            "x": p[2],
            "y": p[3],
            "z": p[4],
            "slot": idx + 1
        }
    return players

def get_shooter_coords(players, shooter_id):
    '''Return the shooter's x/y coordinates if the shooter is available.'''
    
    if shooter_id not in players:
        return None

    return (
        players[shooter_id]["x"],
        players[shooter_id]["y"]
    )

def extract_tracking_features( moments_window,shot_row):
    '''
    Extract tracking-based shot features from the moments before a shot.

    Inputs:
        moments_window: Tracking frames leading up to the shot.
        shot_row: Shot event data containing the shooter information.

    Returns:
        Dictionary with shooter, defender, spacing, speed, and ball features.
    '''

    features = {}
    if len(moments_window) == 0:
        return features

    shooter_id = shot_row["PLAYER_ID"]

    # Use final frame before shot
    final_moment = moments_window[0]

    players = build_player_dict(final_moment)

    if shooter_id not in players:
        return features

    shooter = players[shooter_id]

    sx = shooter["x"]
    sy = shooter["y"]

    shooter_team = shooter["team_id"]

    # Shooter x,y coords
    features["shooter_slot"] = shooter['slot']
    features["shooter_x"] = sx
    features["shooter_y"] = sy
    features["shooter_team_id"] = shooter_team

    # ===================================================
    # 1. Shot angle
    # ===================================================
    dx = basket_x - sx
    dy = basket_y - sy

    features["shot_angle"] = np.degrees(np.arctan2(dy, dx))

    # ===================================================
    # 2. Distance to basket
    # ===================================================
    basket_dist = np.sqrt(dx**2 + dy**2)

    features["distance_to_basket_tracking"] = (basket_dist)

    # ===================================================
    # 3. Nearest defender
    # ===================================================
    defender_distances = []

    for pid, pdata in players.items():

        if pdata["team_id"] == shooter_team:
            continue

        dist = euclidean((sx, sy), (pdata["x"], pdata["y"]))

        defender_distances.append(dist)

    if len(defender_distances) > 0:

        features["nearest_defender_dist"] = min(defender_distances)

        features["avg_defender_dist"] = np.mean(defender_distances)

    # ===================================================
    # 4. Defenders within radius
    # ===================================================
    for radius in [3, 5, 7]:

        count = np.sum(np.array(defender_distances) <= radius)

        features[f"defenders_within_{radius}ft"] = count

    # ===================================================
    # 5. Offensive spacing
    # ===================================================
    # Compute the offensive spacing area via a convex hull
    offensive_players = []

    for pid, pdata in players.items():
        if pdata["team_id"] == shooter_team:
            offensive_players.append(
                [pdata["x"], pdata["y"]]
            )

    if len(offensive_players) >= 3:
        try:
            hull = ConvexHull(offensive_players)
            features["offensive_spacing_area"] = (hull.volume)
        except:
            features["offensive_spacing_area"] = np.nan

    # ===================================================
    # 6. Shooter velocity
    # ===================================================
    if len(moments_window) >= 5:
        shooter_speed = compute_shooter_speed(moments_window[-1], moments_window[0], shooter_id)
        if shooter_speed:
            features["shooter_speed"] = (shooter_speed)
            
        # All player speeds
        player_speeds = compute_player_speeds(moments_window[-1], moments_window[0])
        features.update(player_speeds)

        # Defender closing speed
        features["defender_closing_speed"] = (compute_defender_closing_speed(moments_window,shooter_id))

    # ===================================================
    # 7. Ball height
    # ===================================================
    ball = final_moment[5][0]
    features["ball_height"] = ball[4]

    # Ball speed before shot
    features["ball_speed"] = (compute_ball_speed(moments_window))

    # Ball speed in xy before shot
    features["ball_xy_speed"] = (compute_ball_xy_speed(moments_window))

    return features

# Helper functions for deep learning features

In [10]:
def sample_moments_every_0_2s(moments_window, num_steps=6, step_size=0.2):
    """
    Samples moments at:
    t=0.0, -0.2, -0.4, ..., etc.

    Returns moments ordered from oldest -> newest.
    """

    if len(moments_window) == 0:
        return None

    # Closest moment to shot
    latest_moment = moments_window[0]

    latest_clock = latest_moment[2]

    sampled = []

    for step in range(num_steps):
        target_clock = latest_clock + step * step_size

        closest = min(
            moments_window,
            key=lambda m: abs(float(m[2]) - target_clock)
        )

        times = [moment[2] for moment in moments_window]

        #print(f"Target :{target_clock}, closest {closest[2]}, times: {times}")

        sampled.append(closest)

        
    return sampled

    #for step in reversed(range(num_steps)):
#
    #    target_clock = latest_clock + (step * step_size)
#
    #    closest = min(
    #        moments_window,
    #        key=lambda m: abs(m[2] - target_clock)
    #    )
#
    #    sampled.append(closest)
#
    #return sampled

In [11]:
def euclidean_distance(x1, y1, x2, y2):
    return math.sqrt((x1 - x2) ** 2 + (y1 - y2) ** 2)


def extract_temporal_positions(sampled_moments, shot_row):
    """
    Temporal player representation witho ordering determined ONLY at shot frame (latest frame).
    """

    output = {}

    shooter_id = shot_row["PLAYER_ID"]
    offense_team_id = shot_row["TEAM_ID"]

    # STEP 1: Determine canonical ordering at shot frame
    shot_moment = sampled_moments[-1]
    shot_entities = shot_moment[5]

    shot_players = shot_entities[1:]

    # Find shooter
    shooter = None

    for p in shot_players:
        if p[1] == shooter_id:
            shooter = p
            break

    if shooter is None:
        return output

    sx, sy = shooter[2], shooter[3]

    defenders = []
    attackers = []

    for p in shot_players:
        team_id = p[0]
        player_id = p[1]

        if player_id == shooter_id:
            continue

        px, py = p[2], p[3]
        dist = euclidean_distance(px, py, sx, sy)

        player_info = {
            "player_id": player_id,
            "team_id": team_id,
            "dist": dist
        }

        if team_id == offense_team_id:
            attackers.append(player_info)
        else:
            defenders.append(player_info)

    # Sort only at shot frame
    defenders = sorted(defenders, key=lambda x: x["dist"])
    attackers = sorted(attackers, key=lambda x: x["dist"])

    # Assign IDs based on ordering
    defender_id_map = {
        d["player_id"]: idx + 1 for idx, d in enumerate(defenders)
    }

    attacker_id_map = {
        a["player_id"]: idx + 1 for idx, a in enumerate(attackers)
    }

    # STEP 2: Extract temporal features
    for t_idx, moment in enumerate(sampled_moments):

        postfix = f"_t{t_idx}"

        entities = moment[5]
        players = entities[1:]

        # Player lookup
        player_lookup = {
            p[1]: p
            for p in players
        }

        # SHOOTER
        shooter_player = player_lookup.get(shooter_id)

        if shooter_player is None:
            continue

        sx, sy = shooter_player[2], shooter_player[3]

        output[f"shooter_x{postfix}"] = sx
        output[f"shooter_y{postfix}"] = sy

        # DEFENDERS
        for player_id, canonical_idx in defender_id_map.items():

            p = player_lookup.get(player_id)

            if p is None:
                continue

            px, py = p[2], p[3]

            dx = px - sx
            dy = py - sy

            dist = euclidean_distance(px, py, sx, sy)

            prefix = f"defender{canonical_idx}"

            output[f"{prefix}_dx{postfix}"] = dx
            output[f"{prefix}_dy{postfix}"] = dy
            output[f"{prefix}_dist{postfix}"] = dist

        # ATTACKERS
        for player_id, canonical_idx in attacker_id_map.items():

            p = player_lookup.get(player_id)

            if p is None:
                continue

            px, py = p[2], p[3]

            dx = px - sx
            dy = py - sy

            dist = euclidean_distance(px, py, sx, sy)

            prefix = f"attacker{canonical_idx}"

            output[f"{prefix}_dx{postfix}"] = dx
            output[f"{prefix}_dy{postfix}"] = dy
            output[f"{prefix}_dist{postfix}"] = dist

    return output

# Main Loop

In [12]:
# Loop over all games, extract shot moments and relevant features

merged_rows = []

tracking_files = list(DATA_DIR.glob("*.7z"))

for tracking_path in tqdm(tracking_files):

    try:
        # Extract json
        json_path = extract_7z_file(tracking_path)

        if json_path is None:
            continue

        game = load_tracking_json(json_path)

        game_id = str(game["gameid"])
        game_id = game_id[2:]

        # Load event CSV
        event_csv_path = EVENTS_DIR / f"{str(game["gameid"])}.csv"

        if not event_csv_path.exists():
            print('No events df, skipping')
            continue

        event_df = pd.read_csv(event_csv_path)

        event_df["EVENTNUM"] = event_df["EVENTNUM"].astype(str)

        # Filter shots for this game
        game_shots = shots_df.loc[shots_df["GAME_ID"] == game_id]

        if len(game_shots) == 0:
            print(f'No shots, skipping')
            continue

        # Iterate shots
        for _, shot_row in game_shots.iterrows():
            try:
                game_event_id = str(shot_row["GAME_EVENT_ID"])

                # Find event
                event_idx = int(game_event_id)

                # Find closest moment
                moments_window = get_shot_window(
                    game=game,
                    period=shot_row["PERIOD"],
                    shot_time=shot_row["SHOT_TIME"],
                    window_seconds=1.0
                )
                
                if len(moments_window) == 0:
                    print('No window')
                    continue

                # Tracking features
                #moments_window = [
                #    normalize_moment(m)
                #    for m in moments_window
                #]

                # Sample moments every 0.2 seconds
                sampled_moments = sample_moments_every_0_2s(
                    moments_window,
                    num_steps=6,
                    step_size=0.2
                )
                
                sampled_moments = [
                    normalize_moment(m)
                    for m in sampled_moments
                ]

                closest_moment = sampled_moments[0]
                oldest_moment = sampled_moments[-1]

                tracking_features = extract_tracking_features(
                    sampled_moments,
                    shot_row
                )
                
                temporal_positions = extract_temporal_positions(
                    sampled_moments,
                    shot_row
                )


                ## Position features
                position_data = extract_positions(
                    closest_moment
                )
#
                ## Position features
                #position_data_old = extract_positions(
                #    oldest_moment, postfix='_old'
                #)

                # Event CSV row
                event_row = event_df.loc[
                    event_df["EVENTNUM"] == game_event_id
                ]

                if len(event_row) > 0:
                    event_info = (
                        event_row.iloc[0]
                        .to_dict()
                    )
                else:
                    event_info = {}

                # Merge everything
                merged = {}

                merged.update(tracking_features)

                # shots_fixed
                merged.update(shot_row.to_dict())

                # Add shot time
                merged['tracking_game_clock'] = closest_moment[2]
                merged['tracking_game_clock_old'] = oldest_moment[2]
                
                # event csv
                merged.update(
                    {
                        f"event_{k}": v
                        for k, v in event_info.items()
                    }
                )
                # positions
                merged.update(position_data)
                merged.update(temporal_positions)
                merged_rows.append(merged)

            except Exception as e:
                print(
                    f"Error processing shot: {e}"
                )
        break
    except Exception as e:
        print(
            f"Error processing game {tracking_path}: {e}"
        )

# FINAL DATAFRAME
merged_df = pd.DataFrame(merged_rows)

print(merged_df.shape)

merged_df.head()

  0%|                                                                                                                                   | 0/636 [00:08<?, ?it/s]


(176, 306)


,shooter_slot,shooter_x,shooter_y,shooter_team_id,shot_angle,distance_to_basket_tracking,nearest_defender_dist,avg_defender_dist,defenders_within_3ft,defenders_within_5ft,defenders_within_7ft,offensive_spacing_area,shooter_speed,player1_speed,player2_speed,player3_speed,player4_speed,player5_speed,player6_speed,player7_speed,player8_speed,player9_speed,player10_speed,defender_closing_speed,ball_height,ball_speed,ball_xy_speed,ACTION_TYPE,EVENTTIME,EVENT_TYPE,GAME_DATE,GAME_EVENT_ID,GAME_ID,GRID_TYPE,HTM,LOC_X,LOC_Y,MINUTES_REMAINING,PERIOD,PLAYER_ID,PLAYER_NAME,QUARTER,SECONDS_REMAINING,SHOT_ATTEMPTED_FLAG,SHOT_DISTANCE,SHOT_MADE_FLAG,SHOT_TIME,SHOT_TYPE,SHOT_ZONE_AREA,SHOT_ZONE_BASIC,SHOT_ZONE_RANGE,TEAM_ID,TEAM_NAME,VTM,tracking_game_clock,tracking_game_clock_old,event_GAME_ID,event_EVENTNUM,event_EVENTMSGTYPE,event_EVENTMSGACTIONTYPE,event_PERIOD,event_WCTIMESTRING,event_PCTIMESTRING,event_HOMEDESCRIPTION,event_NEUTRALDESCRIPTION,event_VISITORDESCRIPTION,event_SCORE,event_SCOREMARGIN,event_PERSON1TYPE,event_PLAYER1_ID,event_PLAYER1_NAME,event_PLAYER1_TEAM_ID,event_PLAYER1_TEAM_CITY,event_PLAYER1_TEAM_NICKNAME,event_PLAYER1_TEAM_ABBREVIATION,event_PERSON2TYPE,event_PLAYER2_ID,event_PLAYER2_NAME,event_PLAYER2_TEAM_ID,event_PLAYER2_TEAM_CITY,event_PLAYER2_TEAM_NICKNAME,event_PLAYER2_TEAM_ABBREVIATION,event_PERSON3TYPE,event_PLAYER3_ID,event_PLAYER3_NAME,event_PLAYER3_TEAM_ID,event_PLAYER3_TEAM_CITY,event_PLAYER3_TEAM_NICKNAME,event_PLAYER3_TEAM_ABBREVIATION,ball_x,ball_y,ball_z,player1_team_id,player1_id,player1_x,player1_y,player2_team_id,player2_id,player2_x,player2_y,player3_team_id,player3_id,player3_x,player3_y,player4_team_id,player4_id,player4_x,player4_y,player5_team_id,player5_id,player5_x,player5_y,player6_team_id,player6_id,player6_x,player6_y,player7_team_id,player7_id,player7_x,player7_y,player8_team_id,player8_id,player8_x,player8_y,player9_team_id,player9_id,player9_x,player9_y,player10_team_id,player10_id,player10_x,player10_y,shooter_x_t0,shooter_y_t0,defender1_dx_t0,defender1_dy_t0,defender1_dist_t0,defender2_dx_t0,defender2_dy_t0,defender2_dist_t0,defender3_dx_t0,defender3_dy_t0,defender3_dist_t0,defender4_dx_t0,defender4_dy_t0,defender4_dist_t0,defender5_dx_t0,defender5_dy_t0,defender5_dist_t0,attacker1_dx_t0,attacker1_dy_t0,attacker1_dist_t0,attacker2_dx_t0,attacker2_dy_t0,attacker2_dist_t0,attacker3_dx_t0,attacker3_dy_t0,attacker3_dist_t0,attacker4_dx_t0,attacker4_dy_t0,attacker4_dist_t0,shooter_x_t1,shooter_y_t1,defender1_dx_t1,defender1_dy_t1,defender1_dist_t1,defender2_dx_t1,defender2_dy_t1,defender2_dist_t1,defender3_dx_t1,defender3_dy_t1,defender3_dist_t1,defender4_dx_t1,defender4_dy_t1,defender4_dist_t1,defender5_dx_t1,defender5_dy_t1,defender5_dist_t1,attacker1_dx_t1,attacker1_dy_t1,attacker1_dist_t1,attacker2_dx_t1,attacker2_dy_t1,attacker2_dist_t1,attacker3_dx_t1,attacker3_dy_t1,attacker3_dist_t1,attacker4_dx_t1,attacker4_dy_t1,attacker4_dist_t1,shooter_x_t2,shooter_y_t2,defender1_dx_t2,defender1_dy_t2,defender1_dist_t2,defender2_dx_t2,defender2_dy_t2,defender2_dist_t2,defender3_dx_t2,defender3_dy_t2,defender3_dist_t2,defender4_dx_t2,defender4_dy_t2,defender4_dist_t2,defender5_dx_t2,defender5_dy_t2,defender5_dist_t2,attacker1_dx_t2,attacker1_dy_t2,attacker1_dist_t2,attacker2_dx_t2,attacker2_dy_t2,attacker2_dist_t2,attacker3_dx_t2,attacker3_dy_t2,attacker3_dist_t2,attacker4_dx_t2,attacker4_dy_t2,attacker4_dist_t2,shooter_x_t3,shooter_y_t3,defender1_dx_t3,defender1_dy_t3,defender1_dist_t3,defender2_dx_t3,defender2_dy_t3,defender2_dist_t3,defender3_dx_t3,defender3_dy_t3,defender3_dist_t3,defender4_dx_t3,defender4_dy_t3,defender4_dist_t3,defender5_dx_t3,defender5_dy_t3,defender5_dist_t3,attacker1_dx_t3,attacker1_dy_t3,attacker1_dist_t3,attacker2_dx_t3,attacker2_dy_t3,attacker2_dist_t3,attacker3_dx_t3,attacker3_dy_t3,attacker3_dist_t3,attacker4_dx_t3,attacker4_dy_t3,attacker4_dist_t3,shooter_x_t4,shooter_y_t4,defender1_dx_t4,defender1_dy_t4,defender1_dist_t4,defender2_dx_t4,defender2_dy_t4,defender2_d

In [15]:
game['events'][0]

{'eventId': '1',
 'visitor': {'name': 'Charlotte Hornets',
  'teamid': 1610612766,
  'abbreviation': 'CHA',
  'players': [{'lastname': 'Williams',
    'firstname': 'Marvin',
    'playerid': 101107,
    'jersey': '2',
    'position': 'F'},
   {'lastname': 'Lamb',
    'firstname': 'Jeremy',
    'playerid': 203087,
    'jersey': '3',
    'position': 'G'},
   {'lastname': 'Batum',
    'firstname': 'Nicolas',
    'playerid': 201587,
    'jersey': '5',
    'position': 'G-F'},
   {'lastname': 'Lin',
    'firstname': 'Jeremy',
    'playerid': 202391,
    'jersey': '7',
    'position': 'G'},
   {'lastname': 'Harrison',
    'firstname': 'Aaron',
    'playerid': 1626151,
    'jersey': '9',
    'position': 'G'},
   {'lastname': 'Walker',
    'firstname': 'Kemba',
    'playerid': 202689,
    'jersey': '15',
    'position': 'G'},
   {'lastname': 'Hairston',
    'firstname': 'PJ',
    'playerid': 203798,
    'jersey': '19',
    'position': 'F-G'},
   {'lastname': 'Roberts',
    'firstname': 'Brian',


In [13]:
merged_df.to_csv('merged_tracking_events_v3.csv')

In [14]:
merged_df.isna().sum()

shooter_slot         223
shooter_x            223
shooter_y            223
shooter_team_id      223
shot_angle           223
                    ... 
attacker3_dy_t5      252
attacker3_dist_t5    252
attacker4_dx_t5      321
attacker4_dy_t5      321
attacker4_dist_t5    321
Length: 306, dtype: int64